# Shape Blend Splines — Interactive Demo

**Paper:** Q. Li, *Shape Blend Splines*, Computer-Aided Design, 2011. DOI: [10.1016/j.cad.2011.01.006](https://doi.org/10.1016/j.cad.2011.01.006)  
**Repository:** [QL-UoHull/Shape-Blend-Splines](https://github.com/QL-UoHull/Shape-Blend-Splines)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QL-UoHull/Shape-Blend-Splines/blob/main/notebooks/interactive_shape_blend_demo.ipynb)

---

This notebook demonstrates the **Shape Blend Spline (SBS)** framework:

- Blend any number of simple **parametric shapes** (circle, ellipse, rectangle, star, …) into a smooth complex curve.
- Control the **locality parameter** α to tune how strongly each shape is preserved near its centre parameter.
- Adjust **per-shape weights** interactively with sliders.

The core idea:

$$\mathbf{C}(t) = \sum_{j=0}^{k-1} W_j(t)\, \mathbf{S}_j(t)$$

where $W_j(t)$ are *shape-preserving partition-of-unity* weights and $\mathbf{S}_j$ are the constituent shapes.


In [ ]:
# ── Install dependencies (only needed on Google Colab) ──────────────────────
import sys, subprocess, importlib

def colab_install():
    try:
        import google.colab  # only available on Colab
        subprocess.check_call([sys.executable, "-m", "pip", "install",
                               "numpy", "matplotlib", "ipywidgets", "--quiet"])
        subprocess.check_call([sys.executable, "-m", "pip", "install",
                               "git+https://github.com/QL-UoHull/Shape-Blend-Splines.git",
                               "--quiet"])
        print("✔  Dependencies installed.")
    except ImportError:
        pass  # not on Colab — assume local install

colab_install()

In [ ]:
import sys, os
# Allow running directly from the repo without pip-installing the package
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import numpy as np
import matplotlib.pyplot as plt
from functools import partial

from shape_blend_splines import ShapeBlendSpline, ShapeBlender, blend_two_shapes, blend_shape_series, shape_morph
from shape_blend_splines.shapes import (
    circle_arc, ellipse_arc, superellipse_arc,
    rectangle_arc, star_arc, line_segment, from_control_points
)
from shape_blend_splines.basis import blend_weights

# Use interactive matplotlib backend if available
try:
    %matplotlib inline
except:
    pass

print("✔  Imports OK")

## 1  Two-shape blend

Blend a **circle** and a **star** at different blend factors β and locality values α.


In [ ]:
t = np.linspace(0, 1, 500)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, beta in zip(axes, [0.0, 0.5, 1.0]):
    blender = blend_two_shapes(circle_arc, star_arc, blend=beta)
    pts = blender.evaluate(t)
    c  = circle_arc(t)
    s  = star_arc(t)
    ax.plot(c[:,0], c[:,1], '--', color='gray', lw=1, alpha=0.4, label='Circle')
    ax.plot(s[:,0], s[:,1], ':', color='gray', lw=1, alpha=0.4, label='Star')
    ax.plot(pts[:,0], pts[:,1], color='steelblue', lw=2.5, label=f'Blend β={beta}')
    ax.set_aspect('equal'); ax.set_title(f'β = {beta}'); ax.legend(fontsize=8)
    ax.set_xlabel('x'); ax.set_ylabel('y')
fig.suptitle('Two-shape blend: circle → star', fontsize=13)
fig.tight_layout()
plt.show()

### 1a  Interactive two-shape blend

Use the sliders below to explore different blend factors and locality values.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    fig_tw, ax_tw = plt.subplots(figsize=(5, 5))
    plt.close(fig_tw)   # prevent duplicate display

    t_dense = np.linspace(0, 1, 500)

    def update_two_shape(beta=0.5, locality=1.0):
        ax_tw.clear()
        sbs = ShapeBlendSpline([circle_arc, star_arc], locality=locality)
        # Manually mix using weights: w_circle = 1-beta, w_star = beta
        blender = ShapeBlender([circle_arc, star_arc], weights=[1-beta, beta])
        pts = blender.evaluate(t_dense)
        ax_tw.plot(circle_arc(t_dense)[:,0], circle_arc(t_dense)[:,1],
                   '--', color='gray', alpha=0.35, lw=1, label='Circle')
        ax_tw.plot(star_arc(t_dense)[:,0], star_arc(t_dense)[:,1],
                   ':', color='gray', alpha=0.35, lw=1, label='Star')
        ax_tw.plot(pts[:,0], pts[:,1], color='steelblue', lw=2.5, label='Blend')
        ax_tw.set_aspect('equal')
        ax_tw.set_title(f'Two-shape blend  β={beta:.2f}')
        ax_tw.legend(fontsize=9)
        ax_tw.set_xlabel('x'); ax_tw.set_ylabel('y')
        fig_tw.canvas.draw_idle()
        display(fig_tw)

    widgets.interact(
        update_two_shape,
        beta=widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.02,
                                  description='β (blend)'),
        locality=widgets.FloatSlider(value=1.0, min=0.1, max=6.0, step=0.1,
                                      description='α (locality)'),
    )
except ImportError:
    print("ipywidgets not available — run `pip install ipywidgets` for interactive sliders.")
    blender = ShapeBlender([circle_arc, star_arc], weights=[0.5, 0.5])
    pts = blender.evaluate(t_dense)
    fig, ax = plt.subplots(figsize=(5,5))
    ax.plot(pts[:,0], pts[:,1], color='steelblue', lw=2.5)
    ax.set_aspect('equal'); ax.set_title('50/50 blend'); plt.show()

---
## 2  Multi-shape Shape Blend Spline

Blend **five shapes** using the partition-of-unity framework.  
Adjust the **locality parameter α** to see how strongly each shape is preserved near its centre parameter.


In [ ]:
# Define 5 shapes (all centred at origin, comparable scale)
shapes_5 = [
    partial(circle_arc,      center=(0,0), radius=1.0),
    partial(ellipse_arc,     center=(0,0), a=1.4, b=0.7),
    partial(superellipse_arc,center=(0,0), a=1.1, b=1.1, exponent=4.0),
    partial(rectangle_arc,   center=(0,0), width=1.6, height=1.2),
    partial(star_arc,        center=(0,0), outer_radius=1.2, inner_radius=0.45, n_points=5),
]
labels_5 = ['Circle', 'Ellipse', 'Superellipse (n=4)', 'Rectangle', 'Star']
colors_5 = plt.cm.tab10.colors

t_plot = np.linspace(0, 1, 600)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, alpha in zip(axes, [0.8, 4.0]):
    sbs = blend_shape_series(shapes_5, locality=alpha)
    pts = sbs.evaluate(t_plot)
    for j, (s, lbl) in enumerate(zip(shapes_5, labels_5)):
        sp = s(t_plot)
        ax.plot(sp[:,0], sp[:,1], '--', color=colors_5[j], lw=1, alpha=0.35, label=lbl)
    ax.plot(pts[:,0], pts[:,1], 'k-', lw=2.5, label='SBS blend')
    ax.set_aspect('equal'); ax.set_title(f'5-shape SBS  α = {alpha}')
    ax.legend(fontsize=7, ncol=2); ax.set_xlabel('x'); ax.set_ylabel('y')
fig.suptitle('Multi-shape Shape Blend Spline', fontsize=13)
fig.tight_layout()
plt.show()

### 2a  Interactive multi-shape SBS

Adjust the locality α and per-shape weights with the sliders.


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    out = widgets.Output()

    alpha_slider  = widgets.FloatSlider(value=1.5, min=0.1, max=8.0, step=0.1, description='α (locality)')
    w0 = widgets.FloatSlider(value=1.0, min=0.0, max=3.0, step=0.05, description='Circle')
    w1 = widgets.FloatSlider(value=1.0, min=0.0, max=3.0, step=0.05, description='Ellipse')
    w2 = widgets.FloatSlider(value=1.0, min=0.0, max=3.0, step=0.05, description='Superell.')
    w3 = widgets.FloatSlider(value=1.0, min=0.0, max=3.0, step=0.05, description='Rectangle')
    w4 = widgets.FloatSlider(value=1.0, min=0.0, max=3.0, step=0.05, description='Star')

    def redraw_multi(change=None):
        with out:
            clear_output(wait=True)
            raw_w = np.array([w0.value, w1.value, w2.value, w3.value, w4.value])
            total = raw_w.sum()
            norm_w = raw_w / max(total, 1e-9)

            sbs = blend_shape_series(shapes_5, locality=alpha_slider.value)
            blender = ShapeBlender(shapes_5, weights=raw_w)
            pts_sbs  = sbs.evaluate(t_plot)
            pts_mix  = blender.evaluate(t_plot)

            fig, axes = plt.subplots(1, 2, figsize=(12, 5))

            # Left: SBS with locality
            for j, (s, lbl) in enumerate(zip(shapes_5, labels_5)):
                sp = s(t_plot)
                axes[0].plot(sp[:,0], sp[:,1], '--', color=colors_5[j], lw=1, alpha=0.30)
            axes[0].plot(pts_sbs[:,0], pts_sbs[:,1], 'k-', lw=2.5, label='SBS blend')
            axes[0].set_aspect('equal'); axes[0].set_title(f'SBS  α={alpha_slider.value:.1f}')
            axes[0].set_xlabel('x'); axes[0].set_ylabel('y')

            # Right: uniform weight blend
            for j, (s, lbl) in enumerate(zip(shapes_5, labels_5)):
                sp = s(t_plot)
                axes[1].plot(sp[:,0], sp[:,1], '--', color=colors_5[j], lw=1, alpha=0.30,
                             label=f'{lbl} (w={norm_w[j]:.2f})')
            axes[1].plot(pts_mix[:,0], pts_mix[:,1], 'k-', lw=2.5, label='Weighted blend')
            axes[1].set_aspect('equal'); axes[1].set_title('Uniform weight blend')
            axes[1].legend(fontsize=7, ncol=2); axes[1].set_xlabel('x')

            fig.tight_layout()
            plt.show()

    for slider in [alpha_slider, w0, w1, w2, w3, w4]:
        slider.observe(redraw_multi, names='value')

    redraw_multi()
    display(widgets.VBox([alpha_slider, widgets.HBox([w0, w1]), widgets.HBox([w2, w3, w4])]), out)

except ImportError:
    print("ipywidgets not available — showing static plot instead.")
    blender = ShapeBlender(shapes_5)
    pts = blender.evaluate(t_plot)
    fig, ax = plt.subplots(figsize=(5,5))
    ax.plot(pts[:,0], pts[:,1], 'k-', lw=2.5)
    ax.set_aspect('equal'); ax.set_title('Equal-weight 5-shape blend'); plt.show()

---
## 3  Visualising the blend weights

The **partition-of-unity** property ensures the weights always sum to 1.  
Higher α concentrates each weight function near its centre parameter.


In [ ]:
t_w = np.linspace(0, 1, 400)
centers = np.linspace(0, 1, 5)
labels_w = ['Circle', 'Ellipse', 'Superell.', 'Rectangle', 'Star']

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, alpha in zip(axes, [0.5, 1.5, 5.0]):
    W = blend_weights(t_w, centers, locality=alpha)
    for j in range(5):
        ax.plot(t_w, W[j], color=colors_5[j], lw=2, label=labels_w[j])
    ax.fill_between(t_w, 0, 1, alpha=0.04)
    ax.set_ylim(-0.02, 1.05); ax.set_xlabel('t'); ax.set_ylabel('W_j(t)')
    ax.set_title(f'Blend weights  α = {alpha}')
    if alpha == 0.5:
        ax.legend(fontsize=8, loc='upper right')
fig.suptitle('Partition-of-unity blend weights at different locality α', fontsize=12)
fig.tight_layout()
plt.show()
print("Note: columns always sum to 1.0 (partition of unity)")
print("Max deviation from 1:", np.abs(W.sum(axis=0) - 1).max())

---
## 4  Shape morphing sequence

Smoothly morph from one shape to another by varying the blend parameter β ∈ [0, 1].


In [ ]:
n_frames = 7
betas = np.linspace(0, 1, n_frames)
frames = shape_morph(circle_arc, star_arc, n_frames=n_frames, locality=2.5, n_points=400)

fig, axes = plt.subplots(1, n_frames, figsize=(3.2*n_frames, 3.5))
for ax, pts, beta in zip(axes, frames, betas):
    ax.plot(pts[:,0], pts[:,1], color='steelblue', lw=2.0)
    ax.set_aspect('equal'); ax.set_title(f'β={beta:.2f}', fontsize=10)
    ax.axis('off')
fig.suptitle('Shape morphing: circle → star (β = blend parameter)', fontsize=12)
fig.tight_layout()
plt.show()

---
## 5  Free-form curve from control points

Specify your own (x, y) **control points** and get a smooth Shape Blend Spline automatically.  
Try editing the `control_pts` array in the next cell!


In [ ]:
from shape_blend_splines.curve import ControlPointSpline

# ── Edit these control points ─────────────────────────────────────────────
control_pts = np.array([
    [0.0,  0.0],
    [1.0,  1.5],
    [2.5,  0.4],
    [3.5,  2.0],
    [5.0,  0.0],
    [6.0,  1.2],
])
locality_cp = 2.0   # try 0.5, 1.0, 3.0, …
# ─────────────────────────────────────────────────────────────────────────

sbs_cp = ControlPointSpline(control_pts, locality=locality_cp)
t_cp   = np.linspace(0, 1, 600)
pts_cp = sbs_cp.evaluate(t_cp)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(pts_cp[:,0], pts_cp[:,1], color='steelblue', lw=2.5, label='Shape Blend Spline')
ax.plot(control_pts[:,0], control_pts[:,1], 'o--', color='tomato',
        lw=1.5, markersize=9, label='Control points')
for i, (x, y) in enumerate(control_pts):
    ax.annotate(f'P{i}', (x, y), textcoords='offset points', xytext=(4, 6), fontsize=9)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title(f'Free-form control-point spline  (α = {locality_cp})')
ax.legend(); fig.tight_layout(); plt.show()

### 5a  Interactive control-point editor

Drag the **locality** slider and change the *y* offset with the per-point sliders below.  
(For full click-to-drag interaction, run the notebook locally with `%matplotlib widget`.)


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    out_cp = widgets.Output()

    # Base control points
    base_pts = np.array([
        [0.0, 0.0], [1.0, 1.5], [2.5, 0.4],
        [3.5, 2.0], [5.0, 0.0], [6.0, 1.2],
    ])

    alpha_cp_s = widgets.FloatSlider(value=2.0, min=0.1, max=8.0, step=0.1, description='α')
    dy_sliders = [widgets.FloatSlider(value=base_pts[i,1], min=-3.0, max=4.0, step=0.05,
                                       description=f'P{i}.y') for i in range(len(base_pts))]

    def redraw_cp(change=None):
        with out_cp:
            clear_output(wait=True)
            pts_edit = base_pts.copy()
            for i, sl in enumerate(dy_sliders):
                pts_edit[i, 1] = sl.value
            sbs_e = ControlPointSpline(pts_edit, locality=alpha_cp_s.value)
            t_e   = np.linspace(0, 1, 600)
            curve = sbs_e.evaluate(t_e)

            fig, ax = plt.subplots(figsize=(8, 4))
            ax.plot(curve[:,0], curve[:,1], color='steelblue', lw=2.5, label='SBS curve')
            ax.plot(pts_edit[:,0], pts_edit[:,1], 'o--', color='tomato',
                    lw=1.5, markersize=9, label='Control pts')
            ax.set_xlabel('x'); ax.set_ylabel('y')
            ax.set_title(f'Interactive control-point SBS  α={alpha_cp_s.value:.1f}')
            ax.legend(); ax.set_ylim(-3.5, 5); fig.tight_layout(); plt.show()

    alpha_cp_s.observe(redraw_cp, names='value')
    for sl in dy_sliders:
        sl.observe(redraw_cp, names='value')

    redraw_cp()
    slider_rows = [widgets.HBox(dy_sliders[:3]), widgets.HBox(dy_sliders[3:])]
    display(widgets.VBox([alpha_cp_s] + slider_rows), out_cp)

except ImportError:
    print("ipywidgets not available. Edit `control_pts` manually in the cell above.")

---
## 6  Defining your own shape

Any callable `shape(t) -> np.ndarray of shape (m, 2)` can be used.  
Here we blend a **Lissajous figure** with a **circle**.


In [ ]:
def lissajous(t, a=1.0, b=2.0, delta=np.pi/4):
    """Lissajous figure: x = a*cos(t*2π), y = b*sin(2*t*2π + δ)."""
    theta = t * 2 * np.pi
    x = a * np.cos(theta)
    y = b * np.sin(2 * theta + delta)
    return np.column_stack([x, y])

t_adv = np.linspace(0, 1, 600)
fig, axes = plt.subplots(1, 4, figsize=(15, 4))

for ax, beta in zip(axes, [0.0, 0.33, 0.67, 1.0]):
    blender = ShapeBlender([circle_arc, lissajous], weights=[1-beta, beta])
    pts_adv = blender.evaluate(t_adv)
    ax.plot(circle_arc(t_adv)[:,0],  circle_arc(t_adv)[:,1],
            '--', color='gray', lw=1, alpha=0.3, label='Circle')
    ax.plot(lissajous(t_adv)[:,0],   lissajous(t_adv)[:,1],
            ':', color='gray', lw=1, alpha=0.3, label='Lissajous')
    ax.plot(pts_adv[:,0], pts_adv[:,1], color='darkorchid', lw=2.5, label=f'β={beta:.2f}')
    ax.set_aspect('equal'); ax.set_title(f'β = {beta:.2f}'); ax.legend(fontsize=7)
    ax.set_xlabel('x'); ax.set_ylabel('y')

fig.suptitle('Custom shape blend: circle → Lissajous figure', fontsize=12)
fig.tight_layout()
plt.show()

---
## Summary

| Feature | API |
|---------|-----|
| Two-shape blend | `blend_two_shapes(S_a, S_b, blend=β)` |
| Multi-shape SBS | `ShapeBlendSpline([S1, S2, …], locality=α)` |
| Weight-only blend | `ShapeBlender([S1, S2, …], weights=[w1, w2, …])` |
| Control-point curve | `ControlPointSpline(pts, locality=α)` |
| Shape morphing | `shape_morph(S_a, S_b, n_frames=N)` |

**Key parameter — locality α:**
- Low α (≈ 0.5) → diffuse global blending  
- α = 1 → smooth raised-cosine blending (default)  
- High α (≥ 3) → strong local shape preservation near each centre

**Cite:**  
Q. Li, "Shape Blend Splines", *Computer-Aided Design*, vol. 43, no. 8, 2011.  
[DOI 10.1016/j.cad.2011.01.006](https://doi.org/10.1016/j.cad.2011.01.006)
